# 06 — Error correction: long-run price relationships

`05_asymmetry.ipynb` looked at week-over-week price changes and found that retail prices respond faster to crude price increases than to decreases. That analysis does not tell us whether crude and retail prices also have a stable long-run relationship.

This notebook checks that question in three steps. First, we test whether the price levels are non-stationary. Second, we test whether each pair of prices is cointegrated. Finally, for the links that are cointegrated, we fit an error-correction model (ECM) and test whether deviations from the long-run relationship are corrected at different speeds depending on the direction of the deviation.

`05` asks whether retail responds faster to a new crude price change, while this notebook asks whether prices move back toward their long-run relationship at different speeds.

The same three links are examined: crude -> wholesale, wholesale -> retail, and direct crude -> retail.

In [1]:
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing pyproject.toml is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("no pyproject.toml found in any parent directory")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.asymmetry import DEFAULT_HAC_MAXLAGS, build_design_matrix, fit_distributed_lag, test_asymmetry
from src.cointegration import adf_test, engle_granger, build_ecm_design_matrix, fit_ecm, test_adjustment_asymmetry
from scripts.verify_alignment import PROCESSED_DIR, RETAIL_SERIES_ID, UPSTREAM_SERIES_ID, load_series

DAILY_CRUDE_SERIES_ID = "DCOILWTICO"
WHOLESALE_SERIES_ID = "WGASUSGULF"

retail_full = load_series(PROCESSED_DIR / "retail.csv", RETAIL_SERIES_ID, "weekly-mon")
weekly_crude = load_series(PROCESSED_DIR / "crude.csv", UPSTREAM_SERIES_ID, "weekly-fri")
daily_crude = load_series(PROCESSED_DIR / "crude.csv", DAILY_CRUDE_SERIES_ID, "daily").dropna(subset=["value"]).reset_index(drop=True)
wholesale_full = load_series(PROCESSED_DIR / "spot.csv", WHOLESALE_SERIES_ID, "weekly-fri")

# Same one-row trim 04/05 use: daily crude's first usable close is 2010-01-04, so any weekly
# series starting on or before that has no valid daily_pit lookback for its first row. Applied
# to every link, not just the daily_pit one, so all three share one sample.
cutoff = daily_crude["date"].min()
retail = retail_full[retail_full["date"] > cutoff].reset_index(drop=True)
wholesale = wholesale_full[wholesale_full["date"] > cutoff].reset_index(drop=True)

links = [
    ("crude → wholesale", wholesale, daily_crude, 1, "daily_pit"),
    ("wholesale → retail", retail, wholesale, 6, "weekly"),
    ("crude → retail", retail, weekly_crude, 4, "weekly"),
]

## 1. Are the price levels non-stationary?

`adf_test` runs an Augmented Dickey–Fuller (ADF) test for each series. The null hypothesis is that the series has a unit root and is therefore non-stationary in levels.

We expect to fail to reject the null for these price series. If a series is already stationary in levels, the cointegration framework would not be appropriate for that series.

In [2]:
series_to_test = {
    "retail": retail,
    "wholesale": wholesale,
    "weekly crude": weekly_crude,
    "daily crude": daily_crude,
}

adf_results = pd.DataFrame(adf_test(df.set_index("date")["value"], name) for name, df in series_to_test.items())
adf_results.round(4)

,name,stat,p_value,crit_1pct,crit_5pct,crit_10pct,likely_unit_root
0,retail,-2.5595,0.1017,-3.4380,-2.8649,-2.5686,True
1,wholesale,-2.5226,0.1101,-3.4380,-2.8649,-2.5686,True
2,weekly crude,-2.5831,0.0965,-3.4380,-2.8649,-2.5686,True
3,daily crude,-2.6257,0.0878,-3.4319,-2.8622,-2.5671,True


Failing to reject a unit root in levels is only half of what Engle–Granger needs. The procedure assumes both series are I(1) — non-stationary in levels *and* stationary once differenced. If a series were I(2), differencing once would leave a unit root behind and the cointegration test below would be run on the wrong objects.

The same ADF test is therefore repeated on the first differences. Here we expect the opposite result: a clear rejection.

In [3]:
adf_diff_results = pd.DataFrame(
    adf_test(df.set_index("date")["value"].diff().dropna(), name) for name, df in series_to_test.items()
)
# round(4) would print every one of these as a literal 0.0 — the rejection is many orders of
# magnitude stronger than that, and the point of the table is how far past the threshold it is.
adf_diff_results.assign(p_value=adf_diff_results["p_value"].map("{:.2e}".format)).round(4)

,name,stat,p_value,crit_1pct,crit_5pct,crit_10pct,likely_unit_root
0,retail,-15.5746,1.97e-28,-3.4380,-2.8649,-2.5686,False
1,wholesale,-24.5780,0.00e+00,-3.4380,-2.8649,-2.5686,False
2,weekly crude,-14.9861,1.14e-27,-3.4380,-2.8649,-2.5686,False
3,daily crude,-11.3044,1.28e-20,-3.4319,-2.8622,-2.5671,False


Every differenced series rejects the unit-root null decisively — the largest p-value is 1.97e-28, and wholesale's underflows to zero. None of the level series rejected. Both conditions for I(1) therefore hold, and Engle–Granger is applied to series it is valid for.

Worth noting the level p-values sit between 0.088 and 0.110 — not a borderline case for rejection at 5%, but not far from 10% either. The conclusion rests on the combination of the two tables, not on the level test alone.

## 2. Cointegration test

Each link is tested with `engle_granger`. The null hypothesis is that the two price series are not cointegrated, meaning there is no stable long-run relationship between them.

The test also returns `gamma0` and `gamma1`, which describe the estimated long-run relationship.

In [4]:
eg_by_link = {}
coint_rows = []
for name, downstream, upstream, k, mode in links:
    eg = engle_granger(downstream, upstream, mode=mode)
    eg_by_link[name] = eg
    coint_rows.append(
        {
            "link": name,
            "K": k,
            "mode": mode,
            "coint_stat": eg["coint_stat"],
            "coint_pvalue": eg["coint_pvalue"],
            "gamma0": eg["gamma0"],
            "gamma1": eg["gamma1"],
        }
    )

coint_results = pd.DataFrame(coint_rows)
coint_results["cointegrated"] = coint_results["coint_pvalue"] < 0.05
coint_results.round(4)

,link,K,mode,coint_stat,coint_pvalue,gamma0,gamma1,cointegrated
0,crude → wholesale,1,daily_pit,-4.2921,0.0026,0.1319,1.1673,True
1,wholesale → retail,6,weekly,-2.8062,0.1637,0.9939,0.9567,False
2,crude → retail,4,weekly,-3.5030,0.0321,1.1447,1.1025,True


Two of the three links are cointegrated at the 5% level: crude -> wholesale (p = 0.0026) and crude -> retail (p = 0.0321). Wholesale -> retail is not cointegrated (p = 0.164).

This means that the data support a long-run relationship for the two links involving crude, but not for wholesale -> retail. Since an ECM requires cointegration, the wholesale -> retail link is not included in the ECM analysis.

This is also an important difference from `05`: the short-run asymmetry was strongest in wholesale -> retail, but this link does not show a long-run equilibrium relationship in this sample.

## 3. Error-correction model

`build_ecm_design_matrix` extends the short-run model from `05` by adding the lagged deviation from the estimated long-run relationship.

The equilibrium error is split into positive and negative parts: `u_pos_lag1` captures a positive deviation from the long-run relationship, while `u_neg_lag1` captures a negative deviation. This allows the model to estimate separate adjustment speeds for the two directions.

The model is fitted only for the two cointegrated links

In [5]:
res_ecm_by_link = {}
for name, downstream, upstream, k, mode in links:
    eg = eg_by_link[name]
    if eg["coint_pvalue"] >= 0.05:
        print(f"--- {name}: not cointegrated, ECM skipped ---\n")
        continue
    design = build_ecm_design_matrix(downstream, upstream, K=k, gamma0=eg["gamma0"], gamma1=eg["gamma1"], mode=mode)
    res_ecm = fit_ecm(design)
    res_ecm_by_link[name] = res_ecm
    print(f"--- {name} ---")
    print(res_ecm.summary())
    print()

build_design_matrix: dropped 2 of 865 rows to NaN (differencing + 1 lag(s)); 863 rows remain
--- crude → wholesale ---
                            OLS Regression Results                            
Dep. Variable:               d_retail   R-squared:                       0.443
Model:                            OLS   Adj. R-squared:                  0.439
Method:                 Least Squares   F-statistic:                     75.90
Date:                Fri, 04 Sep 2026   Prob (F-statistic):           5.77e-76
Time:                        18:08:09   Log-Likelihood:                 1009.8
No. Observations:                 863   AIC:                            -2006.
Df Residuals:                     856   BIC:                            -1972.
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-----------

The ECM fits the data about as well as the plain models in `05` (R-squared = 0.44 for crude -> wholesale and 0.51 for crude -> retail). Adding the equilibrium error therefore does not materially change the fit of the short-run part of the model.

**Crude -> wholesale.** The same-week responses are almost identical in the two directions (`d_up_lag0` = 0.770 vs. `d_down_lag0` = 0.802), consistent with the lack of short-run asymmetry found in `05`. Both adjustment coefficients are negative and individually border on significant, indicating movement back toward the long-run relationship. The two adjustment speeds are compared directly in section 4.

**Crude -> retail.** The short-run asymmetry from `05` remains: `d_up_lag0` = 0.708 versus `d_down_lag0` = 0.256. The adjustment coefficients are much smaller (-0.021 and -0.006) and neither is individually significant, so the evidence for error correction is weak in this link.

## 4. Is the speed of adjustment asymmetric?

`λ⁺` is the coefficient on `u_pos_lag1`, representing adjustment after a positive deviation from the long-run relationship. `λ⁻` is the coefficient on `u_neg_lag1`, representing adjustment after a negative deviation.

If both coefficients are negative, deviations tend to move back toward the long-run relationship. The test below asks whether the two adjustment speeds are significantly different.

In [6]:
lambda_rows = []
for name, res_ecm in res_ecm_by_link.items():
    adj = test_adjustment_asymmetry(res_ecm)
    lambda_rows.append(
        {
            "link": name,
            "lambda_pos": res_ecm.params["u_pos_lag1"],
            "lambda_neg": res_ecm.params["u_neg_lag1"],
            "diff (lambda+ minus lambda-)": adj["estimate"],
            "se": adj["se"],
            "ci_lo": adj["ci_lo"],
            "ci_hi": adj["ci_hi"],
            "p_value": adj["p_value"],
        }
    )

lambda_results = pd.DataFrame(lambda_rows)
lambda_results.round(4)

,link,lambda_pos,lambda_neg,diff (lambda+ minus lambda-),se,ci_lo,ci_hi,p_value
0,crude → wholesale,-0.0702,-0.0399,-0.0303,0.0482,-0.1248,0.0641,0.5291
1,crude → retail,-0.0208,-0.0064,-0.0144,0.0256,-0.0646,0.0358,0.5740


Both adjustment coefficients are negative in both links, which is consistent with correction back toward the long-run relationship.

In both cases, `|λ⁺| > |λ⁻|`, so the estimated adjustment from above is faster. However, the difference is not statistically significant: p = 0.53 for crude -> wholesale and p = 0.57 for crude -> retail.

There is therefore no evidence of asymmetric long-run adjustment in either cointegrated link.

## 5. Does the short-run asymmetry survive the ECM?

We re-run the short-run asymmetry test from `05` using the ECM specification. This checks whether the crude → retail result changes after accounting for the long-run relationship.

In [7]:
comparison_rows = []
for name, downstream, upstream, k, mode in links:
    if name not in res_ecm_by_link:
        continue
    res_plain = fit_distributed_lag(build_design_matrix(downstream, upstream, K=k, mode=mode), maxlags=DEFAULT_HAC_MAXLAGS)
    res_ecm = res_ecm_by_link[name]
    for h in sorted({0, 1, k}):
        plain = test_asymmetry(res_plain, K=k, horizon=h)
        ecm = test_asymmetry(res_ecm, K=k, horizon=h)
        comparison_rows.append(
            {
                "link": name,
                "horizon": h,
                "plain_estimate": plain["estimate"],
                "plain_p": plain["p_value"],
                "ecm_estimate": ecm["estimate"],
                "ecm_p": ecm["p_value"],
            }
        )

pd.DataFrame(comparison_rows).round(4)

build_design_matrix: dropped 2 of 865 rows to NaN (differencing + 1 lag(s)); 863 rows remain
build_design_matrix: dropped 5 of 865 rows to NaN (differencing + 4 lag(s)); 860 rows remain


,link,horizon,plain_estimate,plain_p,ecm_estimate,ecm_p
0,crude → wholesale,0,-0.0908,0.4382,-0.0316,0.7921
1,crude → wholesale,1,0.0304,0.8544,0.1410,0.4324
2,crude → retail,0,0.4333,0.0001,0.4515,0.0000
3,crude → retail,1,0.2331,0.0239,0.2777,0.0110
4,crude → retail,4,0.0303,0.8137,0.1288,0.3075


For crude -> retail, the short-run asymmetry remains after adding the equilibrium-error term. The gap is 0.452 at h = 0 (p < 0.001) and 0.278 at h = 1 (p < 0.03), compared with 0.433 and 0.233 in the plain model. At h = 4, the gap remains non-significant.

For crude -> wholesale, the result is unchanged: there is no significant short-run asymmetry in either the plain model or the ECM.

## 6. Short-run versus long-run asymmetry

The short-run and long-run results are not the same.

The short-run asymmetry from `05` is strongest in wholesale -> retail, but this link is not cointegrated in the sample. As a result, there is no long-run equilibrium relationship to use for an ECM on this link.

For the two cointegrated links, the estimated long-run adjustment is slightly faster from above than from below, which is the opposite direction from the short-run rockets-and-feathers result. However, the difference is not statistically significant in either link.

The clearest result from this notebook is therefore a limit on the original claim: the data support a **short-run difference in repricing speed**, but they do not provide evidence of **asymmetric long-run error correction**.

Adding the long-run correction term does not remove the short-run asymmetry found in `05`. It remains significant for crude -> retail at h = 0 and h = 1.